In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import time
from tqdm import tqdm


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [18]:
# Use DETERMINISTIC dataset (lowest template ID wins)
df = pd.read_csv("labeled_logs_BGL_deterministic.csv")

print(f"✓ Loaded {len(df)} logs (DETERMINISTIC)")
print(f"✓ Columns: {list(df.columns)}")
print(f"✓ Unique EventIds: {df['EventId'].nunique()}")
print(f"✓ Anomaly ratio: {df['Label'].sum() / len(df):.2%}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLabel distribution:")
print(df['Label'].value_counts())


✓ Loaded 4747957 logs (DETERMINISTIC)
✓ Columns: ['EventId', 'Label']
✓ Unique EventIds: 99
✓ Anomaly ratio: 7.34%

First 5 rows:
  EventId  Label
0      Q9      0
1      Q9      0
2      Q9      0
3      Q9      0
4      Q9      0

Label distribution:
Label
0    4399497
1     348460
Name: count, dtype: int64


In [19]:
event_vocab = {eid: idx for idx, eid in enumerate(sorted(df["EventId"].unique()))}
df["EventIdx"] = df["EventId"].map(event_vocab)

print(f"✓ Vocabulary size: {len(event_vocab)}")
print(f"✓ Sample mappings: {list(event_vocab.items())[:5]}")

✓ Vocabulary size: 99
✓ Sample mappings: [('Q1', 0), ('Q10', 1), ('Q100', 2), ('Q101', 3), ('Q102', 4)]


In [20]:
block_size = 500
sequence_length = 80

print(f"Block size: {block_size}")
print(f"Sequence length: {sequence_length}")


Block size: 500
Sequence length: 80


In [21]:
block_starts = list(range(0, len(df), block_size))
block_labels = [
    1 if df.iloc[i:i+block_size]["Label"].sum() > 0 else 0
    for i in block_starts
]

print(f"✓ Total blocks: {len(block_starts)}")
print(f"✓ Anomalous blocks: {sum(block_labels)}")
print(f"✓ Normal blocks: {len(block_labels) - sum(block_labels)}")

✓ Total blocks: 9496
✓ Anomalous blocks: 1299
✓ Normal blocks: 8197


In [22]:
from sklearn.model_selection import train_test_split

train_block_starts, test_block_starts = train_test_split(
    block_starts, test_size=0.2, stratify=block_labels, random_state=42
)

print(f"✓ Training blocks: {len(train_block_starts)}")
print(f"✓ Test blocks: {len(test_block_starts)}")


✓ Training blocks: 7596
✓ Test blocks: 1900


In [23]:
class TBirdSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, block_starts, sequence_length=80, label_column="Label"):
        self.df = df
        self.block_starts = block_starts
        self.sequence_length = sequence_length
        self.label_column = label_column
        self.samples = []

        # Precompute valid sequence start indices within each block
        for start in block_starts:
            block = df.iloc[start:start + block_size].reset_index(drop=True)
            block["EventIdx"] = block["EventId"].map(event_vocab)

            for i in range(0, len(block) - sequence_length):
                seq = block["EventIdx"].iloc[i:i + sequence_length].tolist()
                labels = block[label_column].iloc[i:i + sequence_length].tolist()
                label = 1 if 1 in labels else 0
                self.samples.append((seq, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, label = self.samples[idx]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [24]:
train_dataset = TBirdSequenceDataset(df, train_block_starts, sequence_length)
print(f"✓ Training sequences: {len(train_dataset)}")

✓ Training sequences: 3190277


In [25]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
print(f"✓ Training batches: {len(train_loader)}")

✓ Training batches: 49849


In [26]:
class EventSequenceModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dropout=.3),
            num_layers=2
        )
        self.attn = nn.Linear(embed_dim, 1)
        
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(.3),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        batch_size, seq_len = x.size()
        
        x = self.embed(x)  # [batch, seq_len, embed_dim]

        # Add positional encoding (learned embedding)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        pos_emb = self.pos_encoder(positions)  # [batch, seq_len, embed_dim]
        x = x + pos_emb
        
        x = x.permute(1, 0, 2)  # [seq_len, batch, embed_dim]
        x = self.transformer(x)
        attn_weights = torch.softmax(self.attn(x), dim=0)  # [seq_len, batch, 1]
        x = (x * attn_weights).sum(dim=0)  # weighted sum
        
        return self.fc(x)

In [27]:
model = EventSequenceModel(vocab_size=len(event_vocab), embed_dim=64, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("✓ Model initialized")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")

✓ Model initialized
✓ Parameters: 583,619


<python-env>/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [28]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)
    for batch_x, batch_y in loop:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = output.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        
    avg_loss = total_loss / len(train_loader.dataset)
    train_acc = correct / len(train_loader.dataset)

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
    print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.6f}")
    print(f"Epoch {epoch+1} Accuracy: {train_acc:.5f}")

Epoch 1: 100%|████████████████████████████████████████████| 49849/49849 [05:08<00:00, 161.46it/s]


Epoch 1 Loss: 433.7434
Epoch 1 Avg Loss: 0.000136
Epoch 1 Accuracy: 0.99697


Epoch 2: 100%|████████████████████████████████████████████| 49849/49849 [05:06<00:00, 162.63it/s]


Epoch 2 Loss: 284.1809
Epoch 2 Avg Loss: 0.000089
Epoch 2 Accuracy: 0.99792


Epoch 3: 100%|████████████████████████████████████████████| 49849/49849 [05:08<00:00, 161.57it/s]


Epoch 3 Loss: 251.7037
Epoch 3 Avg Loss: 0.000079
Epoch 3 Accuracy: 0.99813


Epoch 4: 100%|████████████████████████████████████████████| 49849/49849 [05:11<00:00, 160.12it/s]


Epoch 4 Loss: 234.8405
Epoch 4 Avg Loss: 0.000074
Epoch 4 Accuracy: 0.99825


Epoch 5: 100%|████████████████████████████████████████████| 49849/49849 [05:12<00:00, 159.37it/s]

Epoch 5 Loss: 227.7550
Epoch 5 Avg Loss: 0.000071
Epoch 5 Accuracy: 0.99828


In [ ]:
import json

# Save model weights
# (shipped checkpoint reproducing Table VIII, BGL row: TableVIII_bgl_realtime_classifier.pth)
torch.save(model.state_dict(), "./models/TableVIII_bgl_realtime_classifier.pth")
print("✓ Model saved: ./models/TableVIII_bgl_realtime_classifier.pth")

# Save vocabulary (CRITICAL FOR REAL-TIME!)
with open("./models/event_vocab_BGL.json", "w") as f:
    json.dump(event_vocab, f)
print(f"✓ Vocabulary saved: {len(event_vocab)} unique events")

# Save training metadata
metadata = {
    "dataset": "BGL",
    "vocab_size": len(event_vocab),
    "sequence_length": sequence_length,
    "block_size": block_size,
    "embed_dim": 64,
    "hidden_dim": 128,
    "num_epochs": 5,
    "final_accuracy": train_acc,
    "total_logs": len(df),
    "training_sequences": len(train_dataset)
}

with open("./models/training_metadata_BGL.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("✓ Metadata saved")

✓ Model saved: ./models/TableVIII_bgl_realtime_classifier.pth
✓ Vocabulary saved: 99 unique events
✓ Metadata saved
